# MMLU-Redux 2.0 벤치마크 평가

양자화된 EXAONE 4.0 1.2B 모델을 **MMLU-Redux 2.0** 데이터셋으로 평가합니다.

- 데이터셋: `edinburgh-dawg/mmlu-redux-2.0`
- 평가 방식: 4지선다 객관식 (A/B/C/D) 로그 확률 기반 추론
- 지표: Accuracy (전체 및 subject별)

In [ ]:
import os
import json
import zipfile
import shutil
import tempfile
from datetime import datetime
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from datasets import load_dataset, get_dataset_config_names
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")

## ⚙️ 설정

`MODEL_SOURCE`를 다음 중 하나로 설정하세요:
- `"dir"` → `./model` 폴더에서 직접 로드
- `"zip"` → zip 파일에서 임시 압축 해제 후 로드

In [ ]:
# ── 모델 소스 ───────────────────────────────────
MODEL_SOURCE = "zip"                          # "dir" or "zip"
MODEL_DIR    = "./model"                      # MODEL_SOURCE=="dir" 일 때
ZIP_PATH     = "./baseline_submit_12.zip"     # MODEL_SOURCE=="zip" 일 때
ZIP_SUBDIR   = "model"                        # zip 내부 모델 폴더명

# ── 평가 설정 ───────────────────────────────────
DATASET_ID  = "edinburgh-dawg/mmlu-redux-2.0"
SPLIT       = "test"     # 'test' / 'validation'
MAX_SAMPLES = None       # None=전체, 정수=샘플 제한 (빠른 테스트: 200)
SUBJECTS    = None       # None=전체, 예: ["abstract_algebra", "astronomy"]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if torch.cuda.is_available() else torch.float32

OUTPUT_DIR = "./mmlu_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Model source : {MODEL_SOURCE}")
print(f"Device       : {DEVICE}")
print(f"Dtype        : {DTYPE}")
print(f"Dataset      : {DATASET_ID} [{SPLIT}]")

## 📦 모델 로드

In [ ]:
_tmp_dir = None

if MODEL_SOURCE == "zip":
    print(f"[INFO] {ZIP_PATH} 압축 해제 중...")
    _tmp_dir = tempfile.mkdtemp(prefix="mmlu_eval_")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(_tmp_dir)
    model_path = os.path.join(_tmp_dir, ZIP_SUBDIR)
    print(f"[INFO] 압축 해제 완료: {model_path}")
elif MODEL_SOURCE == "dir":
    model_path = MODEL_DIR
else:
    raise ValueError("MODEL_SOURCE는 'dir' 또는 'zip' 이어야 합니다")

assert os.path.exists(model_path), f"모델 경로 없음: {model_path}"
print(f"[INFO] 모델 경로: {os.path.abspath(model_path)}")
for f in sorted(os.listdir(model_path)):
    size = os.path.getsize(os.path.join(model_path, f)) / 1e6
    print(f"  {f:40s}  {size:8.1f} MB")

In [ ]:
print("[INFO] 토크나이저 로드 중...")
tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
)
tokenizer.padding_side = "left"

print("[INFO] 모델 로드 중...")
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=DTYPE,
    device_map=DEVICE,
    trust_remote_code=True,
)
model.eval()
n_params = sum(p.numel() for p in model.parameters())
print(f"[INFO] 모델 파라미터: {n_params / 1e9:.3f}B")

## 📊 MMLU-Redux 2.0 데이터셋 로드

세 가지 케이스를 자동으로 처리합니다:
1. subject별 config (57개 config)
2. 단일 스플릿 + `subject` 컬럼
3. 기타 fallback

In [ ]:
# ─────────────────────────────────────────────────────────────────
# MMLU-Redux 2.0 로드 (통합 셀 - 항상 df 를 정의합니다)
# ─────────────────────────────────────────────────────────────────

print(f"[INFO] 데이터셋 조회: {DATASET_ID}")

# Step 1: config 목록 조회
configs = []
try:
    configs = get_dataset_config_names(DATASET_ID)
    configs = [c for c in configs if c != "default"]  # 'default' 제외
    print(f"[INFO] config {len(configs)}개 확인: {configs[:5]}{'...' if len(configs) > 5 else ''}")
except Exception as e:
    print(f"[INFO] config 조회 실패 → 단일 스플릿 시도: {e}")

all_records = []

# ── Case 1: subject별 config 로드 ────────────────────────────────
if configs:
    target_configs = configs if SUBJECTS is None else [c for c in configs if c in SUBJECTS]
    print(f"[INFO] 평가할 subject: {len(target_configs)}개")
    failed = []
    for cfg in tqdm(target_configs, desc="Loading subjects"):
        try:
            ds = load_dataset(DATASET_ID, cfg, split=SPLIT, trust_remote_code=True)
            for row in ds:
                choices = row.get("choices", row.get("options", []))
                answer  = row.get("answer",  row.get("correct_answer", row.get("label", 0)))
                all_records.append({
                    "subject" : cfg,
                    "question": row["question"],
                    "choices" : choices,
                    "answer"  : answer,
                })
        except Exception as e:
            failed.append(cfg)
    if failed:
        print(f"[WARN] 로드 실패 subject {len(failed)}개: {failed[:5]}")

# ── Case 2 & 3: 단일 스플릿 fallback ─────────────────────────────
if not all_records:
    print("[INFO] 단일 스플릿으로 전체 데이터 로드 시도...")
    ds_all = None
    for try_split in [SPLIT, "test", "validation", "train"]:
        try:
            ds_all = load_dataset(DATASET_ID, split=try_split, trust_remote_code=True)
            print(f"[INFO] 스플릿 '{try_split}' 로드 성공")
            break
        except Exception:
            continue
    if ds_all is None:
        raise RuntimeError("[ERROR] 어떤 스플릿으로도 데이터셋을 로드할 수 없습니다.")

    print(f"[INFO] 컬럼: {ds_all.column_names}")
    print(f"[INFO] 샘플 수: {len(ds_all)}")
    print(f"[INFO] 첫 번째 행 미리보기:")
    first = ds_all[0]
    for k, v in first.items():
        print(f"  {k}: {str(v)[:120]}")

    for row in ds_all:
        subject = row.get("subject", row.get("category", row.get("topic", "unknown")))
        choices = row.get("choices", row.get("options", []))
        answer  = row.get("answer",  row.get("correct_answer", row.get("label", 0)))
        all_records.append({
            "subject" : subject,
            "question": row["question"],
            "choices" : choices,
            "answer"  : answer,
        })

# ── DataFrame 생성 ────────────────────────────────────────────────
assert all_records, "[ERROR] 레코드가 0건입니다. 위 오류 메시지를 확인하세요."

df = pd.DataFrame(all_records)

# answer 정규화: 문자열 A/B/C/D → 정수 0/1/2/3
def normalize_answer(ans):
    if isinstance(ans, int):
        return ans
    if isinstance(ans, str):
        mapping = {"A": 0, "B": 1, "C": 2, "D": 3}
        val = ans.strip().upper()
        if val in mapping:
            return mapping[val]
        if val.isdigit():
            return int(val)
        return 0
    try:
        return int(ans)
    except Exception:
        return 0

df["answer"] = df["answer"].apply(normalize_answer)

# choices가 정확히 4개 리스트인 행만 유지
before = len(df)
df = df[df["choices"].apply(lambda x: isinstance(x, list) and len(x) == 4)].reset_index(drop=True)
if len(df) < before:
    print(f"[WARN] choices 불량 행 {before - len(df)}건 제외")

if MAX_SAMPLES is not None:
    df = df.sample(min(MAX_SAMPLES, len(df)), random_state=42).reset_index(drop=True)

print(f"\n[OK] 데이터셋 준비 완료!")
print(f"     총 샘플   : {len(df)}")
print(f"     Subject 수: {df['subject'].nunique()}")
print(f"     answer 분포: {df['answer'].value_counts().sort_index().to_dict()}")
print()
print(df.head(2).to_string())

## 🔍 추론 함수

4개 선택지에 대한 **로그 확률(log-likelihood)**을 비교해 가장 높은 선택지를 예측합니다.

In [ ]:
CHOICE_TOKENS = ["A", "B", "C", "D"]


def build_prompt(question: str, choices: list) -> str:
    """Chat template 없이 순수 텍스트 프롬프트"""
    lines = [f"Question: {question}"]
    for letter, choice in zip(CHOICE_TOKENS, choices):
        lines.append(f"{letter}. {choice}")
    lines.append("Answer:")
    return "\n".join(lines)


def build_chat_prompt(question: str, choices: list) -> str:
    """EXAONE chat template 사용 프롬프트"""
    content = "다음 객관식 문제를 읽고 정답 알파벳(A, B, C, D)만 출력하세요.\n\n"
    content += f"문제: {question}\n"
    for letter, choice in zip(CHOICE_TOKENS, choices):
        content += f"{letter}. {choice}\n"
    content += "\n정답:"
    messages = [{"role": "user", "content": content}]
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
    except Exception:
        return build_prompt(question, choices)


# A/B/C/D 단일 토큰 ID 탐색
choice_token_ids = []
for letter in CHOICE_TOKENS:
    found = False
    for variant in [f" {letter}", letter, f"({letter})", f"{letter}."]:
        ids = tokenizer.encode(variant, add_special_tokens=False)
        if len(ids) == 1:
            choice_token_ids.append(ids[0])
            found = True
            break
    if not found:
        ids = tokenizer.encode(letter, add_special_tokens=False)
        choice_token_ids.append(ids[0])

print(f"[INFO] 선택지 토큰 ID: {dict(zip(CHOICE_TOKENS, choice_token_ids))}")


@torch.no_grad()
def predict_choice(question: str, choices: list) -> int:
    """마지막 토큰 위치의 logit으로 A/B/C/D 중 최댓값 인덱스 반환"""
    prompt = build_chat_prompt(question, choices)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    outputs = model(**inputs)
    logits = outputs.logits[0, -1, :]   # (vocab_size,)
    scores = [logits[tid].item() for tid in choice_token_ids]
    return int(np.argmax(scores))


print("[INFO] 추론 함수 준비 완료")

## 🚀 평가 실행

In [ ]:
predictions = []
errors = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="MMLU-Redux 2.0 평가"):
    try:
        pred = predict_choice(row["question"], row["choices"])
        predictions.append(pred)
    except Exception as e:
        errors.append((idx, str(e)))
        predictions.append(-1)

df["prediction"] = predictions
df["correct"]    = df["prediction"] == df["answer"]

if errors:
    print(f"[WARN] 추론 오류 {len(errors)}건:")
    for idx, err in errors[:5]:
        print(f"  idx={idx}: {err}")

valid_df    = df[df["prediction"] >= 0]
overall_acc = valid_df["correct"].mean() * 100
print(f"\n{'='*50}")
print(f"  전체 정확도: {overall_acc:.2f}%  ({int(valid_df['correct'].sum())}/{len(valid_df)})")
print(f"{'='*50}")

## 📈 결과 분석

In [ ]:
subject_acc = (
    valid_df.groupby("subject")["correct"]
    .agg(["mean", "count", "sum"])
    .rename(columns={"mean": "accuracy", "count": "total", "sum": "correct_cnt"})
    .sort_values("accuracy", ascending=False)
)
subject_acc["accuracy"] = (subject_acc["accuracy"] * 100).round(2)

print("== Subject별 정확도 (상위 20개) ==")
print(subject_acc.head(20).to_string())
print(f"\n== 하위 Subject (하위 10개) ==")
print(subject_acc.tail(10).to_string())

In [ ]:
summary = {
    "timestamp"          : datetime.now().isoformat(),
    "model_path"         : os.path.abspath(model_path),
    "dataset"            : DATASET_ID,
    "split"              : SPLIT,
    "total_samples"      : len(valid_df),
    "overall_accuracy_pct": round(overall_acc, 4),
    "subject_accuracies" : subject_acc["accuracy"].to_dict(),
}

summary_path = os.path.join(OUTPUT_DIR, "mmlu_redux_summary.json")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

detail_path  = os.path.join(OUTPUT_DIR, "mmlu_redux_details.csv")
subject_path = os.path.join(OUTPUT_DIR, "mmlu_redux_by_subject.csv")
df.to_csv(detail_path, index=False, encoding="utf-8-sig")
subject_acc.to_csv(subject_path, encoding="utf-8-sig")

print(f"[INFO] 결과 저장 완료:")
print(f"  요약:      {summary_path}")
print(f"  상세:      {detail_path}")
print(f"  Subject별: {subject_path}")
print(f"\n{'█'*52}")
print(f"  MMLU-Redux 2.0 최종 정확도: {overall_acc:.2f}%")
print(f"{'█'*52}")

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams["font.family"] = "DejaVu Sans"

    top_n     = min(30, len(subject_acc))
    plot_data = pd.concat([subject_acc.head(top_n // 2), subject_acc.tail(top_n // 2)])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))

    ax1.hist(subject_acc["accuracy"], bins=20, color="steelblue", edgecolor="white", alpha=0.85)
    ax1.axvline(overall_acc, color="crimson", linewidth=2, linestyle="--",
                label=f"Overall: {overall_acc:.1f}%")
    ax1.set_xlabel("Accuracy (%)"); ax1.set_ylabel("Subject Count")
    ax1.set_title("MMLU-Redux 2.0\nSubject Accuracy Distribution")
    ax1.legend(); ax1.grid(True, alpha=0.3)

    colors = ["#2ecc71" if a >= 50 else "#e74c3c" for a in plot_data["accuracy"]]
    ax2.barh(range(len(plot_data)), plot_data["accuracy"], color=colors, alpha=0.85)
    ax2.set_yticks(range(len(plot_data)))
    ax2.set_yticklabels(plot_data.index, fontsize=8)
    ax2.axvline(50, color="gray", linestyle=":", linewidth=1)
    ax2.axvline(overall_acc, color="crimson", linewidth=2, linestyle="--",
                label=f"Overall: {overall_acc:.1f}%")
    ax2.set_xlabel("Accuracy (%)")
    ax2.set_title(f"Top/Bottom {top_n // 2} Subjects")
    ax2.legend(); ax2.grid(True, alpha=0.3, axis="x"); ax2.set_xlim(0, 105)

    plt.tight_layout()
    plot_path = os.path.join(OUTPUT_DIR, "mmlu_redux_results.png")
    plt.savefig(plot_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"[INFO] 그래프 저장: {plot_path}")
except ImportError:
    print("[INFO] matplotlib 없음, 시각화 건너뜀")
except Exception as e:
    print(f"[WARN] 시각화 오류: {e}")

In [ ]:
if _tmp_dir and os.path.exists(_tmp_dir):
    shutil.rmtree(_tmp_dir)
    print(f"[INFO] 임시 디렉토리 삭제: {_tmp_dir}")
print("[INFO] 평가 완료!")